# 06 — RL–SBJTS vs RL–Merton/GBM comparator

**Ticket:** `C-RLSBJTS-MERTON-COMP-01`
**Protocol ID:** `c9ef65485a49d40356f3bbb02d491c4b73fcc9ebf0a22f02f64ab87e04a590d4`
**Study mode:** `PRESPECIFIED_ESTIMATION_FIRST_COMPARATIVE_STUDY`
**Confirmatory superiority:** `NOT_CLAIMED`

This notebook adds one scientifically fair **RL–Merton/GBM training-law arm** to the
frozen RL–SBJTS study and estimates, on the **same frozen SBJTS target holdout**, the
difference between the frozen Base 4 SBJTS-target-trained policies and newly trained
Merton/GBM policies that share the same learner, state, constraints, exploration
setting and training budget.

It asks a **model-misspecification / training-environment** question. It does not ask
whether Merton is mathematically wrong inside a GBM world, it is not a pure jump-effect
decomposition, and it carries no external-market-validity claim.

---

## Two run modes, never mixed

```python
RUN_MODE = "SMOKE"      # Claude may execute: tiny budgets, CPU, smoke/ namespace
RUN_MODE = "RESEARCH"   # user executes on the paid Colab NVIDIA T4, research/ namespace
```

`RESEARCH` **hard-requires** a CUDA NVIDIA T4. If `torch.cuda.is_available()` is False,
or the allocated device is not a T4, the notebook raises and stops. There is no CPU
fallback and no NumPy fallback: the frozen Base 4 numerical contract is
`TORCH_CUDA_FLOAT32_BATCHED` and a CPU run would silently change it.

`SMOKE` is forced onto CPU even when a GPU is present, and refuses to write anything
into the research namespace. Every smoke artifact is stamped `SMOKE_EVIDENCE` and is
never scientific evidence.

## Stage map

| Stage | What it does | Blocking |
|---|---|---|
| Step 00 | runtime, artifact resolution by content hash, mode and hardware gate | yes in RESEARCH |
| S0 / U0 | source fingerprint and protocol identity | yes |
| — | entropy time-scaling unit tests (frozen discrete vs continuous-time objective) | yes |
| S1 / U1 | **predeclared two-holdout Base 4 TT reproduction gate** | yes |
| S2 / U2 | empirical Merton/GBM calibration on the frozen training slice only | yes |
| S3 / U3 | bounded Merton-world learner positive control | yes |
| S4 / U4 | Merton arm training, 2 × 40 policies at the frozen budget | — |
| S5 / U5 | Merton arm evaluation on the frozen SBJTS target holdout | — |
| S6 / U6 | analytic exploratory Merton, both exploration conventions | — |
| S7 / U7 | inference: **frozen TT ledger joined with the new MT rows** | needs S1/U1 |

## What this notebook never does

Retrain or overwrite the 80 frozen SBJTS target policies; re-evaluate the SBJTS arm
instead of reading the frozen Base 4 ledger; recalibrate the SBJTS environment; change
the learner mathematics, the state, the wealth accounting, `m`, the action bounds, the
training budget, the holdout namespace or the endpoint definitions; tune anything on
the target holdout; introduce a post-result SESOI; or replace a seed.


## Step 00 — run mode, sources and hardware


In [ ]:
# Step 00 — run mode, artifact resolution, hardware gate
#
# Resumable. Every artifact is located by CONTENT, never by a bare filename: a
# candidate is accepted only when its SHA-256 equals the pinned digest.

RUN_MODE = "SMOKE"          # <-- set to "RESEARCH" for the paid Colab T4 run
ALLOW_NON_T4 = False        # only with a written PMO authorisation; never enables CPU
ALLOW_NON_T4_REASON = None

import base64, hashlib, os, sys, json, shutil

TICKET = "C-RLSBJTS-MERTON-COMP-01"
PROTOCOL_ID = "c9ef65485a49d40356f3bbb02d491c4b73fcc9ebf0a22f02f64ab87e04a590d4"

PINNED = {
    "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8.ipynb":
        "344956031d9e89763370a020d92ec54a669a7cc400e3d2b674de613897c02129",
    "frozen_market_snapshot_U1_BASELINE_4.npz":
        "7e817762849118fc3abf8d4cf98ad8d65d921fa49cb0d1b3bb34d884b73c5b4a",
    "BASE4_05A_FINAL_BUNDLE.zip":
        "77aaf6b2ccdd98d446120b72adaddf01b8819ada12b406a3bc39e073d129ca43",
    "05B_BASE4_SCIENTIFIC_EXPERIMENT_GPU_RESEARCH_v2_0.ipynb":
        "7bb73be0ddb5ad52534e6d2bdf8829fc2603394188d39f0c337b618fedf97657",
    # Frozen Base 4 RESEARCH_GPU outputs, consumed read-only and never rewritten here.
    # Pinned by content because files of the same name also exist in the SMOKE and
    # TORCHCPU output folders and a name-only match could silently pick one of those.
    "policies.npz":
        "34c39a30feb29391684bab03e4cbb0a3ebedd448af645253265413d8406fa294",
    "training_attempts.csv":
        "b56505a2d8d0973e821f0a81c60cfaaf96613c1e72bc3c11e44828f9332729c2",
}
# The frozen Base 4 target-holdout evaluation ledger. Its TT rows ARE the SBJTS arm of
# the comparator: they are read, never regenerated. It is not content-pinned here
# because the research copy is ~31 MB and grew incrementally during the Base 4 run;
# U1 is what establishes that the rows in it are the rows this code reproduces.
FROZEN_EVAL_LEDGER_NAME = "evaluation_results_partial.csv"

SEARCH_ROOTS = [
    "/content/drive/MyDrive/sbjts_rst",
    "/content/drive/MyDrive/sbjts_rst/base4_05b_live/research_outputs_GPU",
    "/content/comparator_inputs",
    os.environ.get("MERTONCOMP_INPUT_DIR", ""),
]
try:
    from google.colab import drive as _gd
    if not os.path.isdir("/content/drive/MyDrive"):
        _gd.mount("/content/drive")
except Exception as _e:
    print("[RUNTIME] Drive not mounted:", type(_e).__name__, _e)


def _sha(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def resolve(name, expected=None, largest_if_ambiguous=False):
    hits = []
    for root in SEARCH_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        for cur, _d, files in os.walk(root):
            if name in files:
                hits.append(os.path.join(cur, name))
    if expected is not None:
        ok = sorted(p for p in hits if _sha(p) == expected)
        if not ok:
            raise RuntimeError(
                f"ARTIFACT_NOT_RESOLVED_BY_CONTENT: {name}; expected sha256 {expected}; "
                f"candidates {hits}")
        return ok[0]
    if not hits:
        raise RuntimeError(f"ARTIFACT_ABSENT: {name}")
    if largest_if_ambiguous and len(hits) > 1:
        # the research ledger is the large one; the smoke/TORCHCPU copies are tiny
        return max(hits, key=os.path.getsize)
    return sorted(hits)[0]


WORK = os.environ.get("MERTONCOMP_WORK",
                      "/content/drive/MyDrive/sbjts_rst/merton_comparator_v1")
if not os.path.isdir(os.path.dirname(WORK)):
    WORK = "/content/merton_comparator_v1"
SP = os.path.join(WORK, "inputs")
EVIDENCE_ROOT = os.path.join(WORK, "evidence")
SRC_DIR = os.path.join(WORK, "comparator_src")
for d in (SP, EVIDENCE_ROOT, SRC_DIR):
    os.makedirs(d, exist_ok=True)

STAGE = {
    "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8.ipynb":
        "03_RL_SBJTS_RESEARCH_GPU_HYBRID_v1_8__driveA.ipynb",
    "frozen_market_snapshot_U1_BASELINE_4.npz":
        "frozen_market_snapshot_U1_BASELINE_4.npz",
    "BASE4_05A_FINAL_BUNDLE.zip": "BASE4_05A_FINAL_BUNDLE.zip",
    "05B_BASE4_SCIENTIFIC_EXPERIMENT_GPU_RESEARCH_v2_0.ipynb": "05B_BASE4_v2_0.ipynb",
    "policies.npz": "base4_policies.npz",
    "training_attempts.csv": "base4_training_attempts.csv",
}
for name, local in STAGE.items():
    dst = os.path.join(SP, local)
    if not os.path.exists(dst):
        shutil.copyfile(resolve(name, PINNED.get(name)), dst)
    if name in PINNED and _sha(dst) != PINNED[name]:
        raise RuntimeError(f"STAGED_ARTIFACT_HASH_MISMATCH: {local}")

FROZEN_TT_LEDGER = os.path.join(SP, "base4_evaluation_results_frozen.csv")
if not os.path.exists(FROZEN_TT_LEDGER):
    shutil.copyfile(resolve(FROZEN_EVAL_LEDGER_NAME, largest_if_ambiguous=True),
                    FROZEN_TT_LEDGER)
print(f"[SOURCES] staged into {SP}; frozen TT ledger "
      f"{os.path.getsize(FROZEN_TT_LEDGER)/1e6:.1f} MB")


## Run-mode, hardware and namespace contract

Two modes that can never be confused: the output root is derived from the mode, RESEARCH refuses to start without a verified CUDA T4, and SMOKE refuses to write under the research namespace.


In [ ]:
_SRC_COMPARATOR_CONFIG_B64 = (
    "IiIiClJ1bi1tb2RlLCBoYXJkd2FyZSBhbmQgbmFtZXNwYWNlIGNvbnRyYWN0IGZvciBDLVJMU0JK"
    "VFMtTUVSVE9OLUNPTVAtMDEuCgpUd28gbW9kZXMsIG5ldmVyIG1peGVkOgoKICAgIFJVTl9NT0RF"
    "ID0gIlNNT0tFIiAgICAgQ2xhdWRlIG1heSBleGVjdXRlLiBUaW55IGJ1ZGdldHMsIHNlcGFyYXRl"
    "IG91dHB1dCBuYW1lc3BhY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIENQVSBwZXJtaXR0"
    "ZWQsIGV2ZXJ5IGFydGlmYWN0IHN0YW1wZWQgU01PS0VfRVZJREVOQ0UuCiAgICBSVU5fTU9ERSA9"
    "ICJSRVNFQVJDSCIgIFVzZXIgZXhlY3V0ZXMgb24gdGhlIHBhaWQgQ29sYWIgTlZJRElBIFQ0LiBG"
    "cm96ZW4gYnVkZ2V0cywKICAgICAgICAgICAgICAgICAgICAgICAgICAgQ1VEQSBmbG9hdDMyIGJh"
    "dGNoZWQgZW5naW5lLCBoYXJkIGhhcmR3YXJlIGdhdGUsIG5vIENQVQogICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICBmYWxsYmFjayBvZiBhbnkga2luZC4KClRoZSBzZXBhcmF0aW9uIGlzIGVuZm9y"
    "Y2VkLCBub3QgbWVyZWx5IGRvY3VtZW50ZWQ6IHRoZSBvdXRwdXQgcm9vdCBpcyBkZXJpdmVkIGZy"
    "b20gdGhlCm1vZGUsIGEgcmVzZWFyY2ggc3RhZ2UgcmVmdXNlcyB0byBzdGFydCB3aXRob3V0IGEg"
    "dmVyaWZpZWQgQ1VEQSBUNCwgYW5kIGEgc21va2Ugc3RhZ2UKcmVmdXNlcyB0byB3cml0ZSBhbnl3"
    "aGVyZSB1bmRlciB0aGUgcmVzZWFyY2ggbmFtZXNwYWNlLgoiIiIKaW1wb3J0IGRhdGV0aW1lCmlt"
    "cG9ydCBqc29uCmltcG9ydCBvcwoKUlVOX01PREVTID0gKCJTTU9LRSIsICJSRVNFQVJDSCIpCgpQ"
    "Uk9GSUxFUyA9IHsKICAgICJTTU9LRSI6IGRpY3QoCiAgICAgICAgcnVuX21vZGU9IlNNT0tFIiwK"
    "ICAgICAgICBldmlkZW5jZV9jbGFzcz0iU01PS0VfRVZJREVOQ0UiLAogICAgICAgIGlzX3NjaWVu"
    "dGlmaWNfZXZpZGVuY2U9RmFsc2UsCiAgICAgICAgb3V0cHV0c19zdWJkaXI9InNtb2tlIiwKICAg"
    "ICAgICAjIFRpbnkgYnVkZ2V0czogZW5vdWdoIHRvIHByb3ZlIHRoZSBwaXBlbGluZSBydW5zLCBy"
    "ZXN1bWVzIGFuZCB3cml0ZXMuCiAgICAgICAgIyBgZXZhbF9wYXRoc2AgYWxvbmUgaXMgaGVsZCBh"
    "dCB0aGUgZnJvemVuIDYwMCBzbyB0aGF0IHRoZSBqb2luIGFnYWluc3QgdGhlCiAgICAgICAgIyBm"
    "cm96ZW4gQmFzZSA0IFRUIGxlZGdlciBpcyBkaW1lbnNpb25hbGx5IGZhaXRoZnVsOyB0aGUgc21v"
    "a2Ugcm93cyBhcmUgc3RpbGwKICAgICAgICAjIHNjaWVudGlmaWNhbGx5IG1lYW5pbmdsZXNzIGJl"
    "Y2F1c2UgdHJhaW5pbmcgaXMgMyB1cGRhdGVzIG9uIDIgcmVwbGljYXRpb25zLgogICAgICAgIHRy"
    "YWluX3BhdGhzPTMyLCBldmFsX3BhdGhzPTYwMCwgdXBkYXRlcz0zLAogICAgICAgIG5fcmVwbGlj"
    "YXRpb25zPTIsIGhvbGRvdXRfZW52X3N0cmVhbXM9MSwgZXZhbF9zZWVkcz0yLAogICAgICAgIGJh"
    "Y2tlbmQ9IlRPUkNIX0NQVV9GTE9BVDMyX0JBVENIRUQiLCBlbmdpbmVfZGV2aWNlPSJjcHUiLAog"
    "ICAgICAgIHJlcXVpcmVzX2N1ZGE9RmFsc2UsIHJlcXVpcmVzX3Q0PUZhbHNlLAogICAgKSwKICAg"
    "ICJSRVNFQVJDSCI6IGRpY3QoCiAgICAgICAgcnVuX21vZGU9IlJFU0VBUkNIIiwKICAgICAgICBl"
    "dmlkZW5jZV9jbGFzcz0iVVNFUl9DT0xBQl9SRVNFQVJDSF9FVklERU5DRSIsCiAgICAgICAgaXNf"
    "c2NpZW50aWZpY19ldmlkZW5jZT1UcnVlLAogICAgICAgIG91dHB1dHNfc3ViZGlyPSJyZXNlYXJj"
    "aCIsCiAgICAgICAgIyBmcm96ZW4gQmFzZSA0IGJ1ZGdldHM7IG5vbmUgb2YgdGhlc2UgbWF5IGJl"
    "IGVkaXRlZAogICAgICAgIHRyYWluX3BhdGhzPTUxMiwgZXZhbF9wYXRocz02MDAsIHVwZGF0ZXM9"
    "NDAwLAogICAgICAgIG5fcmVwbGljYXRpb25zPTQwLCBob2xkb3V0X2Vudl9zdHJlYW1zPTIwLCBl"
    "dmFsX3NlZWRzPTE1LAogICAgICAgIGJhY2tlbmQ9IlRPUkNIX0NVREFfRkxPQVQzMl9CQVRDSEVE"
    "IiwgZW5naW5lX2RldmljZT0iY3VkYTowIiwKICAgICAgICByZXF1aXJlc19jdWRhPVRydWUsIHJl"
    "cXVpcmVzX3Q0PVRydWUsCiAgICApLAp9CgojIFByZWRlY2xhcmVkIEJhc2UgNCByZXByb2R1Y3Rp"
    "b24gc3Vic2V0LiBGaXhlZCBoZXJlLCBiZWZvcmUgYW55IGNvbXBhcmlzb24gaXMgcnVuLCBhbmQK"
    "IyBkZWxpYmVyYXRlbHkgc3Bhbm5pbmcgdHdvIGhvbGRvdXQgc3RyZWFtcyBhcyB0aGUgdGlja2V0"
    "IHJlcXVpcmVzLgpSRVBST0RVQ1RJT05fU1VCU0VUID0gewogICAgImpvaW50X3RyYWluaW5nX3Jl"
    "cGxpY2F0aW9ucyI6ICgwLCAxKSwKICAgICJjb25zdHJhaW50cyI6ICgiTE9OR19PTkxZX0ZVTEwi"
    "LCAiTE9OR19PTkxZX0NBUDUwIiksCiAgICAiaG9sZG91dF9lbnZfc3RyZWFtcyI6ICgwLCAxKSwK"
    "ICAgICJldmFsX3NlZWRzIjogKDAsIDEpLAogICAgImNlbGxfY29kZSI6ICJUVCIsCiAgICAibl9y"
    "b3dzX2V4cGVjdGVkIjogMiAqIDIgKiAyICogMiwKICAgICJwcmVkZWNsYXJlZCI6IFRydWUsCiAg"
    "ICAicmF0aW9uYWxlIjogInRoZSBzbWFsbGVzdCBzdWJzZXQgdGhhdCBleGVyY2lzZXMgYm90aCBj"
    "b25zdHJhaW50IHN0cmF0YSwgdHdvICIKICAgICAgICAgICAgICAgICAiZGlzdGluY3QgaG9sZG91"
    "dCBtYXJrZXQgc3RyZWFtcyBhbmQgdHdvIGRpc3RpbmN0IGV2YWx1YXRpb24gYWN0aW9uICIKICAg"
    "ICAgICAgICAgICAgICAic2VlZHMsIHdoaWNoIGlzIHdoYXQgbWFrZXMgdGhlIGdhdGUgc2Vuc2l0"
    "aXZlIHRvIGEgc2VlZC1uYW1lc3BhY2UgIgogICAgICAgICAgICAgICAgICJvciBtYXJrZXQtZ2Vu"
    "ZXJhdG9yIGRlZmVjdCByYXRoZXIgdGhhbiBvbmx5IHRvIGFyaXRobWV0aWMgZHJpZnQiLAp9CgpF"
    "WFBFQ1RFRF9UNF9TVUJTVFJJTkdTID0gKCJUNCIsKQoKCmRlZiB1dGMoKToKICAgIHJldHVybiBk"
    "YXRldGltZS5kYXRldGltZS5ub3coZGF0ZXRpbWUudGltZXpvbmUudXRjKS5pc29mb3JtYXQoKQoK"
    "CmRlZiBwcm9maWxlKHJ1bl9tb2RlKToKICAgIGlmIHJ1bl9tb2RlIG5vdCBpbiBSVU5fTU9ERVM6"
    "CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVU5LTk9XTl9SVU5fTU9ERToge3J1bl9tb2Rl"
    "IXJ9OyBleHBlY3RlZCBvbmUgb2Yge1JVTl9NT0RFU30iKQogICAgcmV0dXJuIGRpY3QoUFJPRklM"
    "RVNbcnVuX21vZGVdKQoKCmRlZiBvdXRwdXRfcm9vdChldmlkZW5jZV9yb290LCBydW5fbW9kZSk6"
    "CiAgICAiIiJTTU9LRSBhbmQgUkVTRUFSQ0ggY2FuIG5ldmVyIHJlc29sdmUgdG8gdGhlIHNhbWUg"
    "ZGlyZWN0b3J5LiIiIgogICAgcCA9IHByb2ZpbGUocnVuX21vZGUpCiAgICByb290ID0gb3MucGF0"
    "aC5qb2luKGV2aWRlbmNlX3Jvb3QsIHBbIm91dHB1dHNfc3ViZGlyIl0pCiAgICBvcy5tYWtlZGly"
    "cyhyb290LCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIHJvb3QKCgpkZWYgYXNzZXJ0X25hbWVz"
    "cGFjZV9pc29sYXRpb24ocGF0aCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpOgogICAgIiIiUmVm"
    "dXNlIHRvIHdyaXRlIGEgc21va2UgYXJ0aWZhY3QgaW50byB0aGUgcmVzZWFyY2ggbmFtZXNwYWNl"
    "LCBvciB2aWNlIHZlcnNhLiIiIgogICAgcCA9IG9zLnBhdGguYWJzcGF0aChwYXRoKQogICAgc21v"
    "a2UgPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKGV2aWRlbmNlX3Jvb3QsICJzbW9rZSIp"
    "KQogICAgcmVzZWFyY2ggPSBvcy5wYXRoLmFic3BhdGgob3MucGF0aC5qb2luKGV2aWRlbmNlX3Jv"
    "b3QsICJyZXNlYXJjaCIpKQogICAgaWYgcnVuX21vZGUgPT0gIlNNT0tFIiBhbmQgcC5zdGFydHN3"
    "aXRoKHJlc2VhcmNoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJTTU9LRV9XUklURV9J"
    "TlRPX1JFU0VBUkNIX05BTUVTUEFDRV9GT1JCSURERU46IHtwfSIpCiAgICBpZiBydW5fbW9kZSA9"
    "PSAiUkVTRUFSQ0giIGFuZCBwLnN0YXJ0c3dpdGgoc21va2UpOgogICAgICAgIHJhaXNlIFJ1bnRp"
    "bWVFcnJvcihmIlJFU0VBUkNIX1dSSVRFX0lOVE9fU01PS0VfTkFNRVNQQUNFX0ZPUkJJRERFTjog"
    "e3B9IikKICAgIHJldHVybiBUcnVlCgoKZGVmIGhhcmR3YXJlX21hbmlmZXN0KCk6CiAgICAiIiJF"
    "dmVyeXRoaW5nIFBNTyBuZWVkcyB0byBjb25maXJtIHRoZSBydW4gcmVhbGx5IHVzZWQgdGhlIHJl"
    "bnRlZCBUNC4iIiIKICAgIG1hbiA9IHsicmVjb3JkZWRfYXRfdXRjIjogdXRjKCksICJ0b3JjaF9h"
    "dmFpbGFibGUiOiBGYWxzZSwKICAgICAgICAgICAiY3VkYV9hdmFpbGFibGUiOiBGYWxzZSwgImRl"
    "dmljZV9uYW1lIjogTm9uZSwgImRldmljZV9pbmRleCI6IE5vbmUsCiAgICAgICAgICAgImN1ZGFf"
    "cnVudGltZV92ZXJzaW9uIjogTm9uZSwgImN1ZG5uX3ZlcnNpb24iOiBOb25lLAogICAgICAgICAg"
    "ICJ0b3JjaF92ZXJzaW9uIjogTm9uZSwgInRvdGFsX21lbW9yeV9tYiI6IE5vbmUsCiAgICAgICAg"
    "ICAgImNhcGFiaWxpdHkiOiBOb25lLCAiaXNfdDQiOiBGYWxzZX0KICAgIHRyeToKICAgICAgICBp"
    "bXBvcnQgdG9yY2gKICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBleGM6CiAgICAgICAgbWFuWyJp"
    "bXBvcnRfZXJyb3IiXSA9IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9IgogICAgICAgIHJl"
    "dHVybiBtYW4KICAgIG1hblsidG9yY2hfYXZhaWxhYmxlIl0gPSBUcnVlCiAgICBtYW5bInRvcmNo"
    "X3ZlcnNpb24iXSA9IHRvcmNoLl9fdmVyc2lvbl9fCiAgICBtYW5bImN1ZGFfcnVudGltZV92ZXJz"
    "aW9uIl0gPSBnZXRhdHRyKHRvcmNoLnZlcnNpb24sICJjdWRhIiwgTm9uZSkKICAgIHRyeToKICAg"
    "ICAgICBtYW5bImN1ZG5uX3ZlcnNpb24iXSA9IHRvcmNoLmJhY2tlbmRzLmN1ZG5uLnZlcnNpb24o"
    "KQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgIyBub3FhOiBCTEUwMDEKICAgICAgICBtYW5bImN1ZG5uX3ZlcnNpb24iXSA9"
    "IE5vbmUKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcHJvcHMgPSB0"
    "b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcygwKQogICAgICAgIG5hbWUgPSB0b3JjaC5j"
    "dWRhLmdldF9kZXZpY2VfbmFtZSgwKQogICAgICAgIG1hbi51cGRhdGUoY3VkYV9hdmFpbGFibGU9"
    "VHJ1ZSwgZGV2aWNlX2luZGV4PTAsIGRldmljZV9uYW1lPW5hbWUsCiAgICAgICAgICAgICAgICAg"
    "ICB0b3RhbF9tZW1vcnlfbWI9aW50KHByb3BzLnRvdGFsX21lbW9yeSAvIDIgKiogMjApLAogICAg"
    "ICAgICAgICAgICAgICAgY2FwYWJpbGl0eT1mIntwcm9wcy5tYWpvcn0ue3Byb3BzLm1pbm9yfSIs"
    "CiAgICAgICAgICAgICAgICAgICBpc190ND1hbnkocyBpbiBuYW1lLnVwcGVyKCkgZm9yIHMgaW4g"
    "RVhQRUNURURfVDRfU1VCU1RSSU5HUykpCiAgICByZXR1cm4gbWFuCgoKZGVmIHJlcXVpcmVfcmVz"
    "ZWFyY2hfaGFyZHdhcmUoYWxsb3dfbm9uX3Q0PUZhbHNlLCBhbGxvd19ub25fdDRfcmVhc29uPU5v"
    "bmUpOgogICAgIiIiSGFyZCBnYXRlIGZvciBSVU5fTU9ERT0nUkVTRUFSQ0gnLiBUaGVyZSBpcyBu"
    "byBDUFUgZmFsbGJhY2sgcGF0aC4KCiAgICBgYWxsb3dfbm9uX3Q0YCBleGlzdHMgb25seSBzbyBQ"
    "TU8gY2FuIGF1dGhvcmlzZSBhIGRpZmZlcmVudCBDVURBIGRldmljZSBpbiB3cml0aW5nOwogICAg"
    "aXQgY2Fubm90IGJlIHVzZWQgdG8gcnVuIG9uIENQVSwgYW5kIHRoZSBvdmVycmlkZSBhbmQgaXRz"
    "IHJlYXNvbiBhcmUgcmVjb3JkZWQgaW4KICAgIHRoZSBtYW5pZmVzdCB0aGF0IFBNTyBhdWRpdHMu"
    "CiAgICAiIiIKICAgIG1hbiA9IGhhcmR3YXJlX21hbmlmZXN0KCkKICAgIGlmIG5vdCBtYW5bInRv"
    "cmNoX2F2YWlsYWJsZSJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAg"
    "IlJFU0VBUkNIX01PREVfUkVRVUlSRVNfVE9SQ0g6IFB5VG9yY2ggaXMgbm90IGltcG9ydGFibGUu"
    "IFJFU0VBUkNIIG1vZGUgIgogICAgICAgICAgICAiaGFzIG5vIENQVSBvciBOdW1QeSBmYWxsYmFj"
    "ayBieSBkZXNpZ24uIikKICAgIGlmIG5vdCBtYW5bImN1ZGFfYXZhaWxhYmxlIl06CiAgICAgICAg"
    "cmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAiUkVTRUFSQ0hfTU9ERV9SRVFVSVJFU19D"
    "VURBOiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGlzIEZhbHNlLiBUaGUgIgogICAgICAgICAg"
    "ICAiZnJvemVuIEJhc2UgNCBudW1lcmljYWwgY29udHJhY3QgaXMgVE9SQ0hfQ1VEQV9GTE9BVDMy"
    "X0JBVENIRUQgYW5kIHRoaXMgIgogICAgICAgICAgICAidGlja2V0IGZvcmJpZHMgYSBzaWxlbnQg"
    "Q1BVIGZhbGxiYWNrLiBJbiBDb2xhYiBjaG9vc2UgIgogICAgICAgICAgICAiUnVudGltZSA+IENo"
    "YW5nZSBydW50aW1lIHR5cGUgPiBUNCBHUFUgYW5kIHJlLXJ1biBmcm9tIHRoZSB0b3AuIikKICAg"
    "IGlmIG5vdCBtYW5bImlzX3Q0Il0gYW5kIG5vdCBhbGxvd19ub25fdDQ6CiAgICAgICAgcmFpc2Ug"
    "UnVudGltZUVycm9yKAogICAgICAgICAgICBmIlJFU0VBUkNIX01PREVfRVhQRUNUU19OVklESUFf"
    "VDQ6IGFsbG9jYXRlZCBkZXZpY2UgaXMgIgogICAgICAgICAgICBmInttYW5bJ2RldmljZV9uYW1l"
    "J10hcn0uIFRoZSByZXNlYXJjaCBwbGFuLCBiYXRjaCBzaXplcyBhbmQgcnVudGltZSAiCiAgICAg"
    "ICAgICAgICJlc3RpbWF0ZXMgYXJlIHdyaXR0ZW4gZm9yIHRoZSByZW50ZWQgVDQuIFJ1bm5pbmcg"
    "b24gYW5vdGhlciBHUFUgbmVlZHMgYW4gIgogICAgICAgICAgICAiZXhwbGljaXQgUE1PIGF1dGhv"
    "cmlzYXRpb247IHNldCBhbGxvd19ub25fdDQ9VHJ1ZSB3aXRoIGEgd3JpdHRlbiByZWFzb24gIgog"
    "ICAgICAgICAgICAib25seSBhZnRlciBQTU8gcmVjb3JkcyB0aGF0IGRlY2lzaW9uLiIpCiAgICBt"
    "YW5bImFsbG93X25vbl90NF9vdmVycmlkZSJdID0gYm9vbChhbGxvd19ub25fdDQpCiAgICBtYW5b"
    "ImFsbG93X25vbl90NF9yZWFzb24iXSA9IGFsbG93X25vbl90NF9yZWFzb24KICAgIG1hblsiZ2F0"
    "ZSJdID0gIlJFU0VBUkNIX0hBUkRXQVJFX0dBVEVfUEFTUyIKICAgIHJldHVybiBtYW4KCgpkZWYg"
    "cmVzb2x2ZV9iYWNrZW5kKHJ1bl9tb2RlLCBhbGxvd19ub25fdDQ9RmFsc2UsIGFsbG93X25vbl90"
    "NF9yZWFzb249Tm9uZSk6CiAgICAiIiJSZXR1cm4gKGJhY2tlbmQsIGRldmljZSwgaGFyZHdhcmUg"
    "bWFuaWZlc3QpIGZvciB0aGUgbW9kZS4gUkVTRUFSQ0ggbmV2ZXIgZmFsbHMgYmFjay4iIiIKICAg"
    "IHAgPSBwcm9maWxlKHJ1bl9tb2RlKQogICAgaWYgcnVuX21vZGUgPT0gIlJFU0VBUkNIIjoKICAg"
    "ICAgICBtYW4gPSByZXF1aXJlX3Jlc2VhcmNoX2hhcmR3YXJlKGFsbG93X25vbl90NCwgYWxsb3df"
    "bm9uX3Q0X3JlYXNvbikKICAgICAgICByZXR1cm4gcFsiYmFja2VuZCJdLCBwWyJlbmdpbmVfZGV2"
    "aWNlIl0sIG1hbgogICAgbWFuID0gaGFyZHdhcmVfbWFuaWZlc3QoKQogICAgbWFuWyJnYXRlIl0g"
    "PSAiU01PS0VfTk9fSEFSRFdBUkVfUkVRVUlSRU1FTlQiCiAgICAjIFNNT0tFIHN0YXlzIG9uIENQ"
    "VSBldmVuIHdoZW4gYSBHUFUgaGFwcGVucyB0byBiZSBwcmVzZW50LCBzbyB0aGF0IGEgc21va2Ug"
    "cnVuIGNhbgogICAgIyBuZXZlciBiZSBtaXN0YWtlbiBmb3IsIG9yIHNpbGVudGx5IG1lcmdlZCB3"
    "aXRoLCBhIHJlc2VhcmNoIHJ1bi4KICAgIG1hblsic21va2VfZm9yY2VkX2NwdSJdID0gVHJ1ZQog"
    "ICAgcmV0dXJuIHBbImJhY2tlbmQiXSwgcFsiZW5naW5lX2RldmljZSJdLCBtYW4KCgpkZWYgc3Rh"
    "bXAocGF5bG9hZCwgcnVuX21vZGUsIHN0YWdlKToKICAgICIiIkV2ZXJ5IGFydGlmYWN0IGNhcnJp"
    "ZXMgaXRzIG1vZGUsIGV2aWRlbmNlIGNsYXNzIGFuZCBhIGNsYWltLXN0YXR1cyByZW1pbmRlci4i"
    "IiIKICAgIHAgPSBwcm9maWxlKHJ1bl9tb2RlKQogICAgb3V0ID0gewogICAgICAgICJ0aWNrZXQi"
    "OiAiQy1STFNCSlRTLU1FUlRPTi1DT01QLTAxIiwKICAgICAgICAic3RhZ2UiOiBzdGFnZSwKICAg"
    "ICAgICAicnVuX21vZGUiOiBydW5fbW9kZSwKICAgICAgICAiZXZpZGVuY2VfY2xhc3MiOiBwWyJl"
    "dmlkZW5jZV9jbGFzcyJdLAogICAgICAgICJpc19zY2llbnRpZmljX2V2aWRlbmNlIjogcFsiaXNf"
    "c2NpZW50aWZpY19ldmlkZW5jZSJdLAogICAgICAgICJnZW5lcmF0ZWRfYXRfdXRjIjogdXRjKCks"
    "CiAgICB9CiAgICBpZiBydW5fbW9kZSA9PSAiU01PS0UiOgogICAgICAgIG91dFsiY2xhaW1fc3Rh"
    "dHVzIl0gPSAoIlNNT0tFX0VWSURFTkNFIOKAlCBleGVjdXRpb24gcHJvb2Ygb25seS4gVGhpcyBh"
    "cnRpZmFjdCBpcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibm90IGEgc2NpZW50"
    "aWZpYyByZXN1bHQgYW5kIG11c3QgbmV2ZXIgYmUgcmVwb3J0ZWQgYXMgIgogICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAgImNvbXBhcmF0b3IgZXZpZGVuY2UuIikKICAgIG91dC51cGRhdGUo"
    "cGF5bG9hZCkKICAgIHJldHVybiBvdXQKCgpkZWYgd3JpdGVfanNvbihwYXRoLCBwYXlsb2FkLCBy"
    "dW5fbW9kZSwgc3RhZ2UsIGV2aWRlbmNlX3Jvb3QpOgogICAgYXNzZXJ0X25hbWVzcGFjZV9pc29s"
    "YXRpb24ocGF0aCwgcnVuX21vZGUsIGV2aWRlbmNlX3Jvb3QpCiAgICBvcy5tYWtlZGlycyhvcy5w"
    "YXRoLmRpcm5hbWUocGF0aCksIGV4aXN0X29rPVRydWUpCiAgICB0bXAgPSBwYXRoICsgIi50bXAi"
    "CiAgICB3aXRoIG9wZW4odG1wLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAg"
    "anNvbi5kdW1wKHN0YW1wKHBheWxvYWQsIHJ1bl9tb2RlLCBzdGFnZSksIGYsIGluZGVudD0yKQog"
    "ICAgICAgIGYuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGYuZmlsZW5vKCkpCiAgICBqc29uLmxv"
    "YWQob3Blbih0bXAsIGVuY29kaW5nPSJ1dGYtOCIpKQogICAgb3MucmVwbGFjZSh0bXAsIHBhdGgp"
    "CiAgICByZXR1cm4gcGF0aAo="
)
_SRC_COMPARATOR_CONFIG = base64.b64decode(_SRC_COMPARATOR_CONFIG_B64).decode()
open(os.path.join(SRC_DIR, "comparator_config.py"), "w").write(_SRC_COMPARATOR_CONFIG)
print('comparator_config.py staged', len(_SRC_COMPARATOR_CONFIG), 'chars')


## Frozen-source loader

Rebuilds the Base 3 engine/learner namespace and the Base 4 protocol glue from verified artifact bytes. No frozen mathematics is re-implemented: every scientific object is executed from the verified Base 3 source text.


In [ ]:
_SRC_FROZEN_LOADER_B64 = (
    "..."
    "ID0gY3RsCiAgICAgICAgcmV0dXJuIG91dAo="
)
_SRC_FROZEN_LOADER = base64.b64decode(_SRC_FROZEN_LOADER_B64).decode()
open(os.path.join(SRC_DIR, "frozen_loader.py"), "w").write(_SRC_FROZEN_LOADER)
print('frozen_loader.py staged', len(_SRC_FROZEN_LOADER), 'chars')


## Merton/GBM arm

Section 6.1 empirical GBM calibration and its predeclared Monte Carlo moment test, the GBM market batch built through the frozen `build_market_batch`, the Base 4 training body with the market law swapped, and the Section 6.2 analytic exploratory Merton policy at both exploration conventions.


In [ ]:
_SRC_MERTON_ARM_B64 = (
    "..."
)
_SRC_MERTON_ARM = base64.b64decode(_SRC_MERTON_ARM_B64).decode()
open(os.path.join(SRC_DIR, "merton_arm.py"), "w").write(_SRC_MERTON_ARM)
print('merton_arm.py staged', len(_SRC_MERTON_ARM), 'chars')


## Entropy time scaling: frozen discrete objective vs continuous-time objective

Documents and unit-tests `m = lambda * dt` between the frozen Base 3 objective `E[log(W_T/W_0)] + m * sum_t H_t` and the Chau-Nguyen-Nguyen continuous-time objective `E[log W_T] + lambda * integral H_t dt`. Nothing frozen is changed; the factor is common to both comparator arms and cancels from Delta, but it decides which exploration weight the analytic benchmark must be built at.


In [ ]:
_SRC_ENTROPY_TIME_SCALING_B64 = (
    "..."
    "IG1haW4oKQo="
)
_SRC_ENTROPY_TIME_SCALING = base64.b64decode(_SRC_ENTROPY_TIME_SCALING_B64).decode()
open(os.path.join(SRC_DIR, "entropy_time_scaling.py"), "w").write(_SRC_ENTROPY_TIME_SCALING)
print('entropy_time_scaling.py staged', len(_SRC_ENTROPY_TIME_SCALING), 'chars')


## Stages S0/U0 … S7/U7

Identical code in both modes. The SBJTS arm is never retrained and never re-evaluated: its target-holdout rows are read from the frozen Base 4 ledger, and the predeclared two-holdout reproduction gate is what licenses that.


In [ ]:
_SRC_COMPARATOR_STAGES_B64 = (
    "..."
    "Cg=="
)
_SRC_COMPARATOR_STAGES = base64.b64decode(_SRC_COMPARATOR_STAGES_B64).decode()
open(os.path.join(SRC_DIR, "comparator_stages.py"), "w").write(_SRC_COMPARATOR_STAGES)
print('comparator_stages.py staged', len(_SRC_COMPARATOR_STAGES), 'chars')


## SMOKE end-to-end self-test

Sixteen checks that Claude executes and PMO can re-execute: the RESEARCH hardware gate refusing without a T4, namespace isolation, the reproduction gate blocking on a missing reference row, checkpoint/resume skipping completed units, output schemas, and inference refusing to run without the gate. Run it with `smoke_runner.main()` after setting RUN_MODE = "SMOKE".


In [ ]:
_SRC_SMOKE_RUNNER_B64 = (
    "IiIiClNNT0tFIGVuZC10by1lbmQgcnVubmVyIGZvciBDLVJMU0JKVFMtTUVSVE9OLUNPTVAtMDEu"
    "IENsYXVkZSBtYXkgZXhlY3V0ZSB0aGlzLgoKSXQgcHJvdmVzLCBhdCB0aW55IGJ1ZGdldHMgYW5k"
    "IGluIGFuIGlzb2xhdGVkIG5hbWVzcGFjZSwgdGhhdDoKCiAgMS4gdGhlIG5vdGVib29rIGxhdW5j"
    "aGVzIGFuZCByZXNvbHZlcyBpdHMgZnJvemVuIHNvdXJjZXM7CiAgMi4gdGhlIFJFU0VBUkNIIGhh"
    "cmR3YXJlIGdhdGUgcmVmdXNlcyB0byBzdGFydCB3aXRob3V0IGEgQ1VEQSBUNDsKICAzLiBTTU9L"
    "RSBjYW5ub3Qgd3JpdGUgaW50byB0aGUgUkVTRUFSQ0ggbmFtZXNwYWNlOwogIDQuIHRoZSBwcmVk"
    "ZWNsYXJlZCB0d28taG9sZG91dCBCYXNlIDQgcmVwcm9kdWN0aW9uIGdhdGUgcnVucywgYW5kIGJs"
    "b2NrcyBjb3JyZWN0bHkKICAgICB3aGVuIGEgcmVmZXJlbmNlIHJvdyBpcyB1bmF2YWlsYWJsZTsK"
    "ICA1LiB0aGUgZW50cm9weSB0aW1lLXNjYWxpbmcgaWRlbnRpdHkgYmV0d2VlbiB0aGUgZnJvemVu"
    "IGRpc2NyZXRlIG9iamVjdGl2ZSBhbmQgdGhlCiAgICAgQ2hhdS1OZ3V5ZW4tTmd1eWVuIGNvbnRp"
    "bnVvdXMtdGltZSBvYmplY3RpdmUgaG9sZHM7CiAgNi4gdGhlIGVtcGlyaWNhbCBHQk0gY2FsaWJy"
    "YXRpb24gYW5kIGl0cyBNb250ZSBDYXJsbyBtb21lbnQgdGVzdCBydW47CiAgNy4gYSB0aW55IE1l"
    "cnRvbiBwb2xpY3kgdHJhaW5zIGFuZCB0aGUgcG9zaXRpdmUtY29udHJvbCBnYXRlcyBldmFsdWF0"
    "ZTsKICA4LiBhIHRpbnkgZXZhbHVhdGlvbiBjb21wbGV0ZXMgYWdhaW5zdCB0aGUgZnJvemVuIHRh"
    "cmdldCBob2xkb3V0IGdlbmVyYXRvcjsKICA5LiBvdXRwdXQgc2NoZW1hcyBhcmUgYXMgZG9jdW1l"
    "bnRlZDsKIDEwLiBjaGVja3BvaW50L3Jlc3VtZSBza2lwcyBjb21wbGV0ZWQgdW5pdHMgaW5zdGVh"
    "ZCBvZiByZXBlYXRpbmcgdGhlbTsKIDExLiBpbmZlcmVuY2Ugam9pbnMgdGhlIEZST1pFTiBCYXNl"
    "IDQgVFQgbGVkZ2VyIHdpdGggdGhlIG5ldyBNVCByb3dzIHdpdGhvdXQKICAgICByZS1ldmFsdWF0"
    "aW5nIHRoZSBTQkpUUyBhcm0uCgpOb3RoaW5nIGhlcmUgaXMgc2NpZW50aWZpYyBldmlkZW5jZS4g"
    "RXZlcnkgYXJ0aWZhY3QgaXMgc3RhbXBlZCBTTU9LRV9FVklERU5DRS4KIiIiCmltcG9ydCBjc3YK"
    "aW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN5cwppbXBvcnQgdGlt"
    "ZQoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19m"
    "aWxlX18pKSkKaW1wb3J0IGNvbXBhcmF0b3JfY29uZmlnIGFzIENDCmltcG9ydCBjb21wYXJhdG9y"
    "X3N0YWdlcyBhcyBTVAppbXBvcnQgZW50cm9weV90aW1lX3NjYWxpbmcgYXMgRVRTCmltcG9ydCBm"
    "cm96ZW5fbG9hZGVyIGFzIEZMCmltcG9ydCBtZXJ0b25fYXJtIGFzIE1BCgpJTlBVVFMgPSBvcy5l"
    "bnZpcm9uLmdldCgiTUVSVE9OQ09NUF9JTlBVVFMiLCAiaW5wdXRzIikKRVZJREVOQ0VfUk9PVCA9"
    "IG9zLmVudmlyb24uZ2V0KCJNRVJUT05DT01QX0VWSURFTkNFIiwgImV2aWRlbmNlIikKIyBUaGUg"
    "ZnJvemVuIEJhc2UgNCB0YXJnZXQtaG9sZG91dCBsZWRnZXIuIEluIENvbGFiIHRoaXMgaXMgdGhl"
    "IGZ1bGwgfjMxIE1CIHJlc2VhcmNoCiMgbGVkZ2VyOyBpbiBDbGF1ZGUncyBzYW5kYm94IG9ubHkg"
    "aXRzIGZpcnN0IDEgTWlCIHByZWZpeCBpcyByZXRyaWV2YWJsZSwgd2hpY2ggaXMgd2h5CiMgdGhl"
    "IHByZWRlY2xhcmVkIHR3by1ob2xkb3V0IGdhdGUgQkxPQ0tTIGhlcmUgYW5kIG9ubHkgdGhlIG1l"
    "Y2hhbmlzbSBjaGVjayBwYXNzZXMuCkZST1pFTl9UVF9MRURHRVIgPSBvcy5lbnZpcm9uLmdldCgK"
    "ICAgICJNRVJUT05DT01QX0ZST1pFTl9UVF9MRURHRVIiLAogICAgb3MucGF0aC5qb2luKElOUFVU"
    "UywgImJhc2U0X2V2YWx1YXRpb25fcmVzdWx0c19mcm96ZW4uY3N2IikpCgoKZGVmIG1haW4oZnJl"
    "c2g9VHJ1ZSk6CiAgICB0X2FsbCA9IHRpbWUudGltZSgpCiAgICBvdXRfZGlyID0gQ0Mub3V0cHV0"
    "X3Jvb3QoRVZJREVOQ0VfUk9PVCwgIlNNT0tFIikKICAgIGlmIGZyZXNoOgogICAgICAgIGZvciBm"
    "IGluIG9zLmxpc3RkaXIob3V0X2Rpcik6CiAgICAgICAgICAgIGlmIGYuZW5kc3dpdGgoKCIuY3N2"
    "IiwgIi5ucHoiKSkgb3IgZiA9PSAicmVzdW1lX21hbmlmZXN0Lmpzb24iOgogICAgICAgICAgICAg"
    "ICAgb3MucmVtb3ZlKG9zLnBhdGguam9pbihvdXRfZGlyLCBmKSkKICAgIGxvZyA9IHsiY2hlY2tz"
    "IjogW119CgogICAgZGVmIHJlYyhuYW1lLCBvaywgZGV0YWlsKToKICAgICAgICBsb2dbImNoZWNr"
    "cyJdLmFwcGVuZCh7ImNoZWNrIjogbmFtZSwgInBhc3MiOiBib29sKG9rKSwgImRldGFpbCI6IGRl"
    "dGFpbH0pCiAgICAgICAgcHJpbnQoZiIgIFt7J1BBU1MnIGlmIG9rIGVsc2UgJ0ZBSUwnfV0ge25h"
    "bWV9OiB7ZGV0YWlsfSIsIGZsdXNoPVRydWUpCgogICAgIyAtLSAyLiBSRVNFQVJDSCBoYXJkd2Fy"
    "ZSBnYXRlIG11c3QgcmVmdXNlIGhlcmUgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAg"
    "ICB0cnk6CiAgICAgICAgQ0MucmVxdWlyZV9yZXNlYXJjaF9oYXJkd2FyZSgpCiAgICAgICAgcmVj"
    "KCJSRVNFQVJDSF9IQVJEV0FSRV9HQVRFX1JFRlVTRVNfV0lUSE9VVF9UNCIsIEZhbHNlLAogICAg"
    "ICAgICAgICAiZ2F0ZSBkaWQgTk9UIHJhaXNlIG9uIGEgbWFjaGluZSB3aXRob3V0IENVREEiKQog"
    "ICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBleGM6CiAgICAgICAgcmVjKCJSRVNFQVJDSF9IQVJE"
    "V0FSRV9HQVRFX1JFRlVTRVNfV0lUSE9VVF9UNCIsIFRydWUsCiAgICAgICAgICAgIHN0cihleGMp"
    "LnNwbGl0KCI6IilbMF0pCgogICAgIyAtLSAzLiBuYW1lc3BhY2UgaXNvbGF0aW9uIC0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgdHJ5OgogICAg"
    "ICAgIENDLmFzc2VydF9uYW1lc3BhY2VfaXNvbGF0aW9uKG9zLnBhdGguam9pbihFVklERU5DRV9S"
    "T09ULCAicmVzZWFyY2giLCAieC5qc29uIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgIlNNT0tFIiwgRVZJREVOQ0VfUk9PVCkKICAgICAgICByZWMoIlNNT0tFX0NBTk5P"
    "VF9XUklURV9JTlRPX1JFU0VBUkNIX05BTUVTUEFDRSIsIEZhbHNlLCAibm8gZ3VhcmQgZmlyZWQi"
    "KQogICAgZXhjZXB0IFJ1bnRpbWVFcnJvciBhcyBleGM6CiAgICAgICAgcmVjKCJTTU9LRV9DQU5O"
    "T1RfV1JJVEVfSU5UT19SRVNFQVJDSF9OQU1FU1BBQ0UiLCBUcnVlLAogICAgICAgICAgICBzdHIo"
    "ZXhjKS5zcGxpdCgiOiIpWzBdKQoKICAgIGN0eCA9IFNULkNvbnRleHQoIlNNT0tFIiwgSU5QVVRT"
    "LCBFVklERU5DRV9ST09UKQogICAgcmVjKCJTTU9LRV9DT05URVhUX0JVSUxUIiwgVHJ1ZSwKICAg"
    "ICAgICBmImJhY2tlbmQ9e2N0eC5iYWNrZW5kfSBkZXZpY2U9e2N0eC5kZXZpY2V9ICIKICAgICAg"
    "ICBmImJsb2Nrcz17bGVuKGN0eC5ibG9ja3MoKSl9IHJlcHM9e2N0eC5yZXBsaWNhdGlvbnMoKX0i"
    "KQoKICAgICMgLS0gMS4gUzAgc291cmNlIGZpbmdlcnByaW50IC0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBmcCA9IFNULnN0YWdlX2ZpbmdlcnBy"
    "aW50KGN0eCkKICAgIHJlYygiUzBfU09VUkNFX0ZJTkdFUlBSSU5UIiwgZnBbImJhc2UzX2NvZGVf"
    "Y2VsbF9jb25jYXRfbWF0Y2giXQogICAgICAgIGFuZCBub3QgZnBbImJhc2UzX25hdGl2ZV9hc3Rf"
    "Y29tcG9uZW50cyJdWyJtaXNtYXRjaGVzIl0KICAgICAgICBhbmQgZnBbImZyb3plbl9lbnZpcm9u"
    "bWVudCJdWyJ0cmFpbl9zaGEyNTYiXSA9PSBGTC5FWFBFQ1RFRF9UUkFJTl9TSEEyNTYsCiAgICAg"
    "ICAgZiJ0cmFpbl9zaGEgbWF0Y2gsIHtmcFsnYmFzZTNfbmF0aXZlX2FzdF9jb21wb25lbnRzJ11b"
    "J24nXX0gQVNUIGNvbXBvbmVudHMiKQoKICAgICMgLS0gNS4gZW50cm9weSB0aW1lLXNjYWxpbmcg"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBl"
    "dHMgPSBFVFMucnVuX3VuaXRfdGVzdHMoY3R4Lm5zKQogICAgZXRzWyJlbXBpcmljYWxfY3VydmF0"
    "dXJlX2RpYWdub3N0aWMiXSA9IEVUUy5lbXBpcmljYWxfY3VydmF0dXJlX2RpYWdub3N0aWMoCiAg"
    "ICAgICAgY3R4Lm5zLCBtdT0wLjA5MTQ0MTYzODk1NTQ4ODksIHNpZ21hPTAuMjA3NzczNDI4Njkz"
    "Mjc4NykKICAgIGN0eC53cml0ZSgiZW50cm9weV90aW1lX3NjYWxpbmdfdGVzdHMuanNvbiIsIGV0"
    "cywgIkVOVFJPUFlfVElNRV9TQ0FMSU5HIikKICAgIHJlYygiRU5UUk9QWV9USU1FX1NDQUxJTkdf"
    "VEVTVFMiLCBldHNbInN0YXR1cyJdLmVuZHN3aXRoKCJQQVNTIiksCiAgICAgICAgZiJ7ZXRzWydz"
    "dGF0dXMnXX07IGxhbWJkYV9lcXVpdj17ZXRzWydzY2FsaW5nJ11bJ2xhbWJkYV9lcXVpdmFsZW50"
    "X29mX2Zyb3plbl9tJ119IikKCiAgICAjIC0tIDRhLiBwcmVkZWNsYXJlZCB0d28taG9sZG91dCBn"
    "YXRlOiBibG9ja2luZyBicmFuY2ggLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZ2F0ZSA9"
    "IFNULnN0YWdlX3JlcHJvZHVjdGlvbihjdHgsIEZST1pFTl9UVF9MRURHRVIpCiAgICByZWMoIlMx"
    "X1BSRURFQ0xBUkVEX1RXT19IT0xET1VUX0dBVEVfQkxPQ0tTX09OX01JU1NJTkdfUkVGRVJFTkNF"
    "IiwKICAgICAgICBnYXRlWyJzdGF0dXMiXSA9PSAiQkxPQ0tFRF9SRVBST0RVQ1RJT05fUkVGRVJF"
    "TkNFX1JPV1NfTUlTU0lORyIKICAgICAgICBhbmQgbGVuKGdhdGVbInJlZmVyZW5jZV9yb3dzX21p"
    "c3NpbmciXSkgPT0gOCwKICAgICAgICBmIntnYXRlWydzdGF0dXMnXX07IHtnYXRlWydyb3dzX2Nv"
    "bXBhcmVkJ119L3tnYXRlWydyb3dzX2V4cGVjdGVkJ119IHJvd3MsICIKICAgICAgICBmIntsZW4o"
    "Z2F0ZVsncmVmZXJlbmNlX3Jvd3NfbWlzc2luZyddKX0gcmVmZXJlbmNlIHJvd3MgdW5hdmFpbGFi"
    "bGUgbG9jYWxseSIpCgogICAgIyAtLSA0Yi4gZ2F0ZSBtZWNoYW5pc20gb24gdGhlIHJvd3MgdGhh"
    "dCBhcmUgYXZhaWxhYmxlIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIG1lY2ggPSBTVC5z"
    "dGFnZV9yZXByb2R1Y3Rpb24oCiAgICAgICAgY3R4LCBGUk9aRU5fVFRfTEVER0VSLAogICAgICAg"
    "IG1lY2hhbmlzbV9jaGVja19zdWJzZXQ9eyJob2xkb3V0X2Vudl9zdHJlYW1zIjogKDAsKX0sCiAg"
    "ICAgICAgb3V0X25hbWU9InJlcHJvZHVjdGlvbl9tZWNoYW5pc21fY2hlY2suanNvbiIpCiAgICBy"
    "ZWMoIlMxX0dBVEVfTUVDSEFOSVNNX1BBU1NFU19PTl9BVkFJTEFCTEVfUk9XUyIsCiAgICAgICAg"
    "bWVjaFsic3RhdHVzIl0uc3RhcnRzd2l0aCgiQkFTRTRfVEFSR0VUX1JFUFJPRFVDVElPTl9QQVNT"
    "IikKICAgICAgICBhbmQgbWVjaFsiYXR0ZW1wdF9pZHNfYWxsX21hdGNoIl0gYW5kIG5vdCBtZWNo"
    "WyJpc19wcmVkZWNsYXJlZF9nYXRlIl0sCiAgICAgICAgZiJ7bWVjaFsnc3RhdHVzJ119OyB7bWVj"
    "aFsncm93c19jb21wYXJlZCddfSByb3dzOyB3b3JzdCBhYnMgIgogICAgICAgIGYie21heCh2Wydt"
    "YXhfYWJzX2RpZmYnXSBmb3IgdiBpbiBtZWNoWydwZXJfZW5kcG9pbnRfd29yc3QnXS52YWx1ZXMo"
    "KSk6LjJlfSIpCgogICAgIyAtLSA2LiBjYWxpYnJhdGlvbiAtLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGNhbCA9IFNULnN0YWdl"
    "X2NhbGlicmF0aW9uKGN0eCkKICAgIHJlYygiUzJfTUVSVE9OX0dCTV9DQUxJQlJBVElPTiIsIGNh"
    "bFsic3RhdHVzIl0uZW5kc3dpdGgoIlBBU1MiKQogICAgICAgIGFuZCBjYWxbImxlYWthZ2VfY29u"
    "dHJvbCJdWyJ0cmFpbmluZ19zbGljZV9zaGEyNTZfbWF0Y2giXSwKICAgICAgICBmIntjYWxbJ3N0"
    "YXR1cyddfTsgc2lnbWFfTT17Y2FsWydjYWxpYnJhdGlvbiddWydzaWdtYV9NJ106LjZmfSwgIgog"
    "ICAgICAgIGYiel9tZWFuPXtjYWxbJ21vbnRlX2NhcmxvX21vbWVudF90ZXN0J11bJ3pfbWVhbidd"
    "OisuMmZ9IikKCiAgICAjIC0tIDcuIHBvc2l0aXZlIGNvbnRyb2wgKyB0cmFpbmluZyAtLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgcGMgPSBTVC5zdGFnZV9w"
    "b3NpdGl2ZV9jb250cm9sKGN0eCwgY2FsWyJjYWxpYnJhdGlvbiJdKQogICAgcmVjKCJTM19QT1NJ"
    "VElWRV9DT05UUk9MX0dBVEVTX0VWQUxVQVRFIiwKICAgICAgICBhbGwoayBpbiByWyJnYXRlcyJd"
    "IGZvciByIGluIHBjWyJyZXN1bHRzIl0KICAgICAgICAgICAgZm9yIGsgaW4gKCJQQzFfUExVTUJJ"
    "TkciLCAiUEM1X0FTQ0VOVCIpKSwKICAgICAgICBmIntwY1snc3RhdHVzJ119OyB7bGVuKHBjWydy"
    "ZXN1bHRzJ10pfSBwb2xpY2llcywgIgogICAgICAgIGYiYWxsX3Bhc3M9e2FsbChyWydhbGxfZ2F0"
    "ZXNfcGFzcyddIGZvciByIGluIHBjWydyZXN1bHRzJ10pfSIpCgogICAgdHIxID0gU1Quc3RhZ2Vf"
    "dHJhaW4oY3R4LCBjYWxbImNhbGlicmF0aW9uIl0sIG1heF9yZXBsaWNhdGlvbnM9MiwgcHJvZ3Jl"
    "c3M9RmFsc2UpCiAgICByZWMoIlM0X1RSQUlOSU5HX1BBUlRJQUwiLCB0cjFbInRyYWluZWRfdGhp"
    "c19ydW4iXSA9PSAyLAogICAgICAgIGYie3RyMVsndHJhaW5lZF90aGlzX3J1biddfSB0cmFpbmVk"
    "LCB7dHIxWydjb21wbGV0ZWRfdG90YWwnXX0ve3RyMVsncGxhbm5lZCddfSB0b3RhbCIpCgogICAg"
    "IyAtLSAxMC4gcmVzdW1lOiBzZWNvbmQgY2FsbCBtdXN0IHNraXAgdGhlIGNvbXBsZXRlZCB1bml0"
    "cyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIHRyMiA9IFNULnN0YWdlX3RyYWluKGN0eCwgY2Fs"
    "WyJjYWxpYnJhdGlvbiJdLCBwcm9ncmVzcz1GYWxzZSkKICAgIHJlYygiUzRfUkVTVU1FX1NLSVBT"
    "X0NPTVBMRVRFRF9VTklUUyIsCiAgICAgICAgdHIyWyJ0cmFpbmVkX3RoaXNfcnVuIl0gPT0gdHIy"
    "WyJwbGFubmVkIl0gLSAyCiAgICAgICAgYW5kIHRyMlsiY29tcGxldGVkX3RvdGFsIl0gPT0gdHIy"
    "WyJwbGFubmVkIl0sCiAgICAgICAgZiJyZXN1bWVkIGFuZCB0cmFpbmVkIHt0cjJbJ3RyYWluZWRf"
    "dGhpc19ydW4nXX0gb2Yge3RyMlsncGxhbm5lZCddfSwgIgogICAgICAgIGYibm9uZSByZXBlYXRl"
    "ZCIpCgogICAgIyAtLSA4LiBldmFsdWF0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIGV2MSA9IFNULnN0YWdlX2V2YWx1"
    "YXRlKGN0eCwgY2FsWyJjYWxpYnJhdGlvbiJdLCBtYXhfYmxvY2tzPTEsIHByb2dyZXNzPUZhbHNl"
    "KQogICAgZXYyID0gU1Quc3RhZ2VfZXZhbHVhdGUoY3R4LCBjYWxbImNhbGlicmF0aW9uIl0sIHBy"
    "b2dyZXNzPUZhbHNlKQogICAgbl9leHAgPSB0cjJbInBsYW5uZWQiXSAqIGxlbihjdHguYmxvY2tz"
    "KCkpCiAgICByZWMoIlM1X0VWQUxVQVRJT05fQU5EX1JFU1VNRSIsCiAgICAgICAgZXYyWyJhdHRl"
    "bXB0cyJdID09IG5fZXhwIGFuZCBldjFbImJsb2Nrc192aXNpdGVkIl0gPT0gMSwKICAgICAgICBm"
    "IntldjJbJ2F0dGVtcHRzJ119L3tuX2V4cH0gTWVydG9uIGF0dGVtcHRzIG92ZXIge2xlbihjdHgu"
    "YmxvY2tzKCkpfSBibG9ja3M7ICIKICAgICAgICBmInJlc3VtZSB2aXNpdGVkIHtldjJbJ2Jsb2Nr"
    "c192aXNpdGVkJ119IHJlbWFpbmluZyBibG9jayhzKSIpCiAgICByZWMoIlM1X1NCSlRTX0FSTV9O"
    "T1RfUkVfRVZBTFVBVEVEIiwKICAgICAgICBhbGwoclsiYXJtIl0gPT0gIk1FUlRPTl9NVCIgZm9y"
    "IHIgaW4KICAgICAgICAgICAgY3N2LkRpY3RSZWFkZXIob3BlbihjdHgucGF0aCgiZXZhbHVhdGlv"
    "bl9hdHRlbXB0cy5jc3YiKSkpKSwKICAgICAgICAiZXZhbHVhdGlvbiBsZWRnZXIgY29udGFpbnMg"
    "b25seSB0aGUgTUVSVE9OX01UIGFybSIpCgogICAgIyAtLSA5LiBzY2hlbWEgLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAg"
    "IHNjaGVtYV9vayA9IFRydWUKICAgIHNjaGVtYV9kZXRhaWwgPSBbXQogICAgZm9yIG5hbWUsIGNv"
    "bHMgaW4gKCgidHJhaW5pbmdfYXR0ZW1wdHMuY3N2IiwgU1QuVFJBSU5fQ09MUyksCiAgICAgICAg"
    "ICAgICAgICAgICAgICAgKCJldmFsdWF0aW9uX2F0dGVtcHRzLmNzdiIsIFNULkVWQUxfQ09MUyks"
    "CiAgICAgICAgICAgICAgICAgICAgICAgKCJhbmFseXRpY19tZXJ0b25fcmVzdWx0cy5jc3YiLCBT"
    "VC5BTkFMWVRJQ19DT0xTKSk6CiAgICAgICAgZ290ID0gbmV4dChpdGVyKGNzdi5yZWFkZXIob3Bl"
    "bihjdHgucGF0aChuYW1lKSkpKSkKICAgICAgICBvayA9IGdvdCA9PSBjb2xzCiAgICAgICAgc2No"
    "ZW1hX29rICY9IG9rCiAgICAgICAgc2NoZW1hX2RldGFpbC5hcHBlbmQoZiJ7bmFtZX06eydvaycg"
    "aWYgb2sgZWxzZSAnTUlTTUFUQ0gnfSIpCiAgICByZWMoIk9VVFBVVF9TQ0hFTUFTIiwgc2NoZW1h"
    "X29rLCAiLCAiLmpvaW4oc2NoZW1hX2RldGFpbCkpCgogICAgIyAtLSAxMS4gaW5mZXJlbmNlIGFn"
    "YWluc3QgdGhlIEZST1pFTiBUVCBsZWRnZXIgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0t"
    "LS0tCiAgICBpbmYgPSBTVC5zdGFnZV9pbmZlcmVuY2UoY3R4LCBGUk9aRU5fVFRfTEVER0VSLCBt"
    "ZWNoWyJzdGF0dXMiXSkKICAgIHNyYyA9IGluZlsic2JqdHNfYXJtX3NvdXJjZSJdCiAgICByZWMo"
    "IlM3X0lORkVSRU5DRV9SRVVTRVNfRlJPWkVOX1RUX0xFREdFUiIsCiAgICAgICAgc3JjWyJtb2Rl"
    "Il0gPT0gIkZST1pFTl9MRURHRVJfUkVVU0UiIGFuZCBub3Qgc3JjWyJyZXRyYWluZWQiXQogICAg"
    "ICAgIGFuZCBub3Qgc3JjWyJyZV9ldmFsdWF0ZWQiXSBhbmQgc3JjWyJyb3dzX2pvaW5lZCJdID4g"
    "MAogICAgICAgIGFuZCBub3Qgc3JjWyJyb3dzX21pc3NpbmciXSwKICAgICAgICBmIntzcmNbJ3Jv"
    "d3Nfam9pbmVkJ119IGZyb3plbiBUVCByb3dzIGpvaW5lZCwgMCBtaXNzaW5nLCAiCiAgICAgICAg"
    "ZiJ7bGVuKGluZlsnZXN0aW1hbmRzJ10pfSBlc3RpbWFuZHMiKQoKICAgICMgLS0gaW5mZXJlbmNl"
    "IG11c3QgcmVmdXNlIHdoZW4gdGhlIGdhdGUgaGFzIG5vdCBwYXNzZWQgLS0tLS0tLS0tLS0tLS0t"
    "LS0tLS0tLS0tLQogICAgdHJ5OgogICAgICAgIFNULnN0YWdlX2luZmVyZW5jZShjdHgsIEZST1pF"
    "Tl9UVF9MRURHRVIsICJCTE9DS0VEX1JFUFJPRFVDVElPTiIpCiAgICAgICAgcmVjKCJTN19SRUZV"
    "U0VTX1dJVEhPVVRfUkVQUk9EVUNUSU9OX0dBVEUiLCBGYWxzZSwgIm5vIGd1YXJkIGZpcmVkIikK"
    "ICAgIGV4Y2VwdCBSdW50aW1lRXJyb3IgYXMgZXhjOgogICAgICAgIHJlYygiUzdfUkVGVVNFU19X"
    "SVRIT1VUX1JFUFJPRFVDVElPTl9HQVRFIiwgVHJ1ZSwgc3RyKGV4Yykuc3BsaXQoIjoiKVswXSkK"
    "CiAgICBsb2cudXBkYXRlKHsKICAgICAgICAicnVuX21vZGUiOiAiU01PS0UiLAogICAgICAgICJl"
    "dmlkZW5jZV9jbGFzcyI6ICJTTU9LRV9FVklERU5DRSIsCiAgICAgICAgImlzX3NjaWVudGlmaWNf"
    "ZXZpZGVuY2UiOiBGYWxzZSwKICAgICAgICAiYWxsX3Bhc3MiOiBhbGwoY1sicGFzcyJdIGZvciBj"
    "IGluIGxvZ1siY2hlY2tzIl0pLAogICAgICAgICJuX2NoZWNrcyI6IGxlbihsb2dbImNoZWNrcyJd"
    "KSwKICAgICAgICAiZWxhcHNlZF9zIjogcm91bmQodGltZS50aW1lKCkgLSB0X2FsbCwgMSksCiAg"
    "ICAgICAgInByb2ZpbGUiOiBjdHgucHJvZmlsZSwKICAgICAgICAiaGFyZHdhcmUiOiBjdHguaGFy"
    "ZHdhcmUsCiAgICAgICAgImNsYWltX3N0YXR1cyI6ICJOT1RfVEVTVEVEIOKAlCBzbW9rZSBvbmx5"
    "LiBObyBjb21wYXJhdG9yIGNsYWltIGlzIHN1cHBvcnRlZCBieSAiCiAgICAgICAgICAgICAgICAg"
    "ICAgICAgICJhbnkgYXJ0aWZhY3QgaW4gdGhpcyBkaXJlY3RvcnkuIiwKICAgIH0pCiAgICBjdHgu"
    "d3JpdGUoInNtb2tlX3JlcG9ydC5qc29uIiwgbG9nLCAiU01PS0VfRU5EX1RPX0VORCIpCiAgICBw"
    "cmludChqc29uLmR1bXBzKHsiYWxsX3Bhc3MiOiBsb2dbImFsbF9wYXNzIl0sICJuX2NoZWNrcyI6"
    "IGxvZ1sibl9jaGVja3MiXSwKICAgICAgICAgICAgICAgICAgICAgICJlbGFwc2VkX3MiOiBsb2db"
    "ImVsYXBzZWRfcyJdfSwgaW5kZW50PTIpKQogICAgcmV0dXJuIGxvZwoKCmlmIF9fbmFtZV9fID09"
    "ICJfX21haW5fXyI6CiAgICByID0gbWFpbigpCiAgICBzeXMuZXhpdCgwIGlmIHJbImFsbF9wYXNz"
    "Il0gZWxzZSAxKQo="
)
_SRC_SMOKE_RUNNER = base64.b64decode(_SRC_SMOKE_RUNNER_B64).decode()
open(os.path.join(SRC_DIR, "smoke_runner.py"), "w").write(_SRC_SMOKE_RUNNER)
print('smoke_runner.py staged', len(_SRC_SMOKE_RUNNER), 'chars')


## Step 00b — mode and hardware gate

`RESEARCH` raises here unless a CUDA NVIDIA T4 is allocated. There is no CPU fallback.


In [ ]:
# Step 00b — mode and hardware gate. RESEARCH stops here without a CUDA T4.
sys.path.insert(0, SRC_DIR) if SRC_DIR not in sys.path else None
import importlib
import comparator_config as CC
importlib.reload(CC)

PROFILE = CC.profile(RUN_MODE)
BACKEND, ENGINE_DEVICE, HARDWARE = CC.resolve_backend(
    RUN_MODE, allow_non_t4=ALLOW_NON_T4, allow_non_t4_reason=ALLOW_NON_T4_REASON)
OUT = CC.output_root(EVIDENCE_ROOT, RUN_MODE)
CC.write_json(os.path.join(OUT, "hardware_manifest.json"),
              {"hardware": HARDWARE, "backend": BACKEND,
               "engine_device": ENGINE_DEVICE, "profile": PROFILE,
               "work_dir": WORK, "inputs_dir": SP, "output_dir": OUT},
              RUN_MODE, "STEP_00_HARDWARE", EVIDENCE_ROOT)
print(json.dumps({"RUN_MODE": RUN_MODE, "backend": BACKEND,
                  "engine_device": ENGINE_DEVICE,
                  "device_name": HARDWARE.get("device_name"),
                  "is_t4": HARDWARE.get("is_t4"),
                  "evidence_class": PROFILE["evidence_class"],
                  "output_dir": OUT}, indent=2))
if RUN_MODE == "SMOKE":
    print("\n[SMOKE] Tiny budgets, CPU, isolated namespace. Nothing written by this "
          "run is scientific evidence.")
else:
    print("\n[RESEARCH] CUDA float32 batched engine on "
          f"{HARDWARE['device_name']} ({HARDWARE['total_memory_mb']} MB). "
          "Frozen Base 4 numerical contract preserved.")


## Run the stages


In [ ]:
# Stages. Identical code in both modes; only the budget profile, backend and output
# namespace differ. Each stage is resumable and writes its own evidence file.
import comparator_stages as ST
import entropy_time_scaling as ETS
importlib.reload(ST); importlib.reload(ETS)

CTX = ST.Context(RUN_MODE, SP, EVIDENCE_ROOT,
                 allow_non_t4=ALLOW_NON_T4, allow_non_t4_reason=ALLOW_NON_T4_REASON)

# --- S0/U0 source fingerprint -----------------------------------------------------
FP = ST.stage_fingerprint(CTX)
print("U0 train_sha match:", FP["frozen_environment"]["train_sha256"][:16], "...",
      "| AST components:", FP["base3_native_ast_components"]["n"],
      "| mismatches:", FP["base3_native_ast_components"]["mismatches"])

# --- entropy time-scaling unit tests ----------------------------------------------
ETS_OUT = ETS.run_unit_tests(CTX.ns)
ETS_OUT["empirical_curvature_diagnostic"] = ETS.empirical_curvature_diagnostic(
    CTX.ns, mu=0.0914416389554889, sigma=0.2077734286932787)
CTX.write("entropy_time_scaling_tests.json", ETS_OUT, "ENTROPY_TIME_SCALING")
if not ETS_OUT["status"].endswith("PASS"):
    raise RuntimeError(ETS_OUT["status"])
print("entropy time scaling:", ETS_OUT["status"],
      "| lambda equivalent of frozen m =",
      ETS_OUT["scaling"]["lambda_equivalent_of_frozen_m"])

# --- S1/U1 predeclared two-holdout Base 4 reproduction gate -----------------------
GATE = ST.stage_reproduction(CTX, FROZEN_TT_LEDGER)
print("U1:", GATE["status"], "|", GATE["rows_compared"], "/", GATE["rows_expected"],
      "rows | attempt ids match:", GATE["attempt_ids_all_match"])
if not GATE["status"].startswith("BASE4_TARGET_REPRODUCTION_PASS"):
    raise RuntimeError(
        f"{GATE['status']} — stop here. Do not train the Merton arm and do not join "
        "the frozen TT ledger until this gate passes.")

# --- S2/U2 empirical Merton/GBM calibration ---------------------------------------
CAL = ST.stage_calibration(CTX)
if not CAL["status"].endswith("PASS"):
    raise RuntimeError(CAL["status"])
print("U2:", CAL["status"], "| sigma_M =", round(CAL["calibration"]["sigma_M"], 6),
      "| mu_M - r_f =", round(CAL["calibration"]["mu_M_minus_r_f"], 6))

# --- S3/U3 bounded Merton-world positive control ----------------------------------
PC = ST.stage_positive_control(
    CTX, CAL["calibration"],
    n_replications=2 if RUN_MODE == "RESEARCH" else None)
if not PC["status"].endswith("PASS"):
    raise RuntimeError(PC["status"])
print("U3:", PC["status"])

# --- S4/U4 Merton arm training ----------------------------------------------------
TR = ST.stage_train(CTX, CAL["calibration"])
print("U4:", TR)

# --- S5/U5 + S6/U6 evaluation on the frozen SBJTS target holdout ------------------
EV = ST.stage_evaluate(CTX, CAL["calibration"])
print("U5/U6:", EV)

# --- S7/U7 inference: frozen TT ledger joined with the new MT rows ----------------
EST = ST.stage_inference(CTX, FROZEN_TT_LEDGER, GATE["status"])
for p in EST["estimands"]:
    if p["tier"] == "CO_PRIMARY":
        print(f"  {p['constraint']:<16} {p['endpoint']:<26} "
              f"Delta={p['estimate']:+.6g}  95% CI [{p['ci_low']:+.6g}, "
              f"{p['ci_high']:+.6g}]")
print("\nSBJTS arm source:", EST["sbjts_arm_source"]["mode"],
      "| rows joined:", EST["sbjts_arm_source"]["rows_joined"],
      "| retrained:", EST["sbjts_arm_source"]["retrained"])
print("CLAIM STATUS: NOT_TESTED until PMO audits these outputs.")


## Colab T4 run instructions — one pass

1. **Runtime → Change runtime type → T4 GPU**, then **Connect**. Confirm with
   `!nvidia-smi` that a Tesla T4 is allocated.
2. Make sure Drive holds the frozen inputs (the notebook finds them by content hash
   under `MyDrive/sbjts_rst`): the Base 3 notebook `v1_8`, the market snapshot, the 05A
   bundle, the Base 4 `v2_0` notebook, and from `base4_05b_live/research_outputs_GPU/`
   the `policies.npz`, `training_attempts.csv` and `evaluation_results_partial.csv`.
3. In **Step 00** set `RUN_MODE = "RESEARCH"`. Leave `ALLOW_NON_T4 = False`.
4. **Runtime → Run all.** Step 00b stops immediately if the device is not a CUDA T4.
5. Expect roughly: U0–U3 a few minutes; U4 the 80-policy training; U5 the 24,000
   Merton evaluations over 300 blocks; U6/U7 minutes. Everything checkpoints, so a
   disconnect is recovered by re-running from the top with the same
   `MERTONCOMP_WORK` directory — completed units are skipped, never repeated.
6. When it finishes, copy everything from `<WORK>/evidence/research/` into the repo at
   `rl_sbjts/evidence/merton_comparator_v1/research/` and commit. That directory's
   `README_EXPECTED_OUTPUTS.md` lists exactly which files must appear.

If a gate raises — `BLOCKED_REPRODUCTION…`, `MERTON_GBM_CALIBRATION_FAIL`,
`LEARNER_POSITIVE_CONTROL_FAIL` — **stop and report it**. Do not edit a threshold, a
budget or a seed to get past it; every one of those thresholds comes from the frozen
sources rather than from this ticket.

## Claim status

`CL-RL-006` remains `NOT_TESTED`. Smoke artifacts are never comparator evidence, and
the research outputs are evidence only after PMO audits them.
